# 04 · Gold: modelo estrella

**Objetivo:** construir el modelo que consume Power BI: **una tabla de hechos** alimentada con CDF y **tres dimensiones**.

                       gold_dim_fecha
                             │ fecha_key
    gold_dim_aerolinea ── gold_fact_vuelo ── gold_dim_aeropuerto
         aerolinea_icao      │          origen_icao / destino_icao

| Tabla | Grano | Cómo se construye |
|---|---|---|
| `gold_fact_vuelo` | 1 fila por vuelo | **CDF de silver + MERGE** (incremental) |
| `gold_dim_aerolinea` | 1 fila por aerolínea | Códigos de silver + catálogo propio (CSV) |
| `gold_dim_aeropuerto` | 1 fila por aeropuerto | Códigos de silver + OurAirports (CSV público) |
| `gold_dim_fecha` | 1 fila por día | Calendario generado |

In [0]:
%run ./00_Setup

## Paso 1 · Catálogos
- **Aeropuertos y países:** dataset público OurAirports (se descarga si falta).
- **Aerolíneas:** catálogo propio con los códigos ICAO que aparecen en los datos. Es un archivo plano en RAW.

In [0]:
if not Path(f"{RAW_CATALOGOS}/countries.csv").exists():
    download_catalogos()

aerolineas_csv = """aerolinea_icao,aerolinea,pais,tipo
TAM,LATAM Airlines Brasil,Brasil,Nacional
GLO,Gol Linhas Aéreas,Brasil,Nacional
AZU,Azul Linhas Aéreas,Brasil,Nacional
PTB,Voepass,Brasil,Nacional
LAN,LATAM Airlines Chile,Chile,Internacional
LPE,LATAM Airlines Perú,Perú,Internacional
CMP,Copa Airlines,Panamá,Internacional
ARG,Aerolíneas Argentinas,Argentina,Internacional
AVA,Avianca,Colombia,Internacional
AAL,American Airlines,Estados Unidos,Internacional
UAL,United Airlines,Estados Unidos,Internacional
DAL,Delta Air Lines,Estados Unidos,Internacional
AFR,Air France,Francia,Internacional
KLM,KLM,Países Bajos,Internacional
DLH,Lufthansa,Alemania,Internacional
BAW,British Airways,Reino Unido,Internacional
IBE,Iberia,España,Internacional
TAP,TAP Air Portugal,Portugal,Internacional
AEA,Air Europa,España,Internacional
UAE,Emirates,Emiratos Árabes Unidos,Internacional
QTR,Qatar Airways,Catar,Internacional
ETH,Ethiopian Airlines,Etiopía,Internacional
FDX,FedEx Express,Estados Unidos,Carga
UPS,UPS Airlines,Estados Unidos,Carga
OTROS,Aviación general / privados,-,Otro
"""
dbutils.fs.put(f"{RAW_CATALOGOS}/aerolineas.csv", aerolineas_csv, overwrite=True)
display(dbutils.fs.ls(RAW_CATALOGOS))

## Paso 2 · Construir el modelo
`build_gold()` = `build_dims()` + `build_fact()`. Lee el código de la tabla de hechos: es el mismo patrón CDF + MERGE de silver.

In [0]:
print(inspect.getsource(build_fact))

In [0]:
build_gold()

for t in (FACT, DIM_AEROLINEA, DIM_AEROPUERTO, DIM_FECHA):
    print(f"{t:<22} {spark.table(t).count():>7} filas")
display(spark.table(STATE))

In [0]:
build_fact()   # segunda vez: nada que hacer

## Paso 3 · Integridad del modelo
Toda clave de la tabla de hechos debe existir en su dimensión, y gold debe tener los mismos vuelos que silver.

In [0]:
f = spark.table(FACT)

def huerfanos(col_fact, dim, col_dim):
    """Claves de la tabla de hechos que no existen en la dimensión."""
    d = spark.table(dim).select(F.col(col_dim).alias(col_fact))   # una sola referencia, mismo nombre de columna
    return f.join(d, on=col_fact, how="left_anti").count()

checks = {
    "Vuelos en fact = vuelos en silver":  f.count() == spark.table(SILVER).count(),
    "vuelo_id único en fact":             f.count() == f.select("vuelo_id").distinct().count(),
    "Huérfanos aerolínea":                huerfanos("aerolinea_icao", DIM_AEROLINEA, "aerolinea_icao") == 0,
    "Huérfanos aeropuerto origen":        huerfanos("origen_icao", DIM_AEROPUERTO, "aeropuerto_icao") == 0,
    "Huérfanos aeropuerto destino":       huerfanos("destino_icao", DIM_AEROPUERTO, "aeropuerto_icao") == 0,
    "Huérfanos fecha":                    huerfanos("fecha_key", DIM_FECHA, "fecha_key") == 0,
}
for regla, ok in checks.items():
    print("✔" if ok else "✘", regla)

In [0]:
# ¿Quedó algo "Sin catálogo"? Son candidatos para agregar al CSV de aerolíneas
display(spark.table(FACT).join(spark.table(DIM_AEROLINEA), "aerolinea_icao")
        .where("aerolinea = 'Sin catálogo'")
        .groupBy("aerolinea_icao").count().orderBy(F.col("count").desc()))

👀 **Qué observar:** todas las reglas deben salir con ✔. Si aparecen aerolíneas "Sin catálogo" con muchos vuelos,
agrégalas al CSV del Paso 1 y vuelve a correr `build_dims()`.

## Paso 4 · Preguntas de negocio (adelanto de Power BI)

In [0]:
spark.sql(f"USE SCHEMA {SCHEMA}")
display(spark.sql(f"""
    -- 1. Participación de mercado por aerolínea
    SELECT a.aerolinea, a.tipo, count(*) AS vuelos,
           round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM {FACT} f JOIN {DIM_AEROLINEA} a USING (aerolinea_icao)
    GROUP BY ALL ORDER BY vuelos DESC LIMIT 10
"""))

In [0]:
display(spark.sql(f"""
    -- 2. Vuelos promedio por día de la semana (¿cae el fin de semana?)
    SELECT d.dia_semana_num, d.dia_semana,
           round(count(*) / count(DISTINCT d.fecha), 0) AS vuelos_promedio_dia
    FROM {FACT} f JOIN {DIM_FECHA} d USING (fecha_key)
    GROUP BY ALL ORDER BY d.dia_semana_num
"""))

In [0]:
display(spark.sql(f"""
    -- 3. Salidas por hora del día desde los aeropuertos estudiados
    SELECT f.hora_salida, count(*) AS salidas
    FROM {FACT} f JOIN {DIM_AEROPUERTO} o ON f.origen_icao = o.aeropuerto_icao
    WHERE o.es_aeropuerto_estudiado
    GROUP BY ALL ORDER BY f.hora_salida
"""))

In [0]:
display(spark.sql(f"""
    -- 4. Top rutas (solo vuelos con ambos extremos conocidos)
    SELECT o.ciudad AS origen, d.ciudad AS destino, d.ambito,
           count(*) AS vuelos, round(avg(f.duracion_min), 0) AS duracion_promedio_min
    FROM {FACT} f
    JOIN {DIM_AEROPUERTO} o ON f.origen_icao  = o.aeropuerto_icao
    JOIN {DIM_AEROPUERTO} d ON f.destino_icao = d.aeropuerto_icao
    WHERE f.origen_icao <> 'DESC' AND f.destino_icao <> 'DESC'
    GROUP BY ALL ORDER BY vuelos DESC LIMIT 15
"""))

## ✔ Checkpoint
- [ ] `gold_fact_vuelo` se construyó con CDF de silver + MERGE (y la segunda corrida no hizo nada)
- [ ] Las tres dimensiones existen y no hay claves huérfanas
- [ ] Fact tiene los mismos vuelos que silver
- [ ] Las 4 consultas responden preguntas de negocio

**Siguiente:** Job (orquestación) y Power BI